In [ ]:
import os
import math
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pickle
%matplotlib inline
from IPython.display import clear_output

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# data_folder = "drive/MyDrive/mestrado_final_files/data/"
# models_folder = "drive/MyDrive/mestrado_final_files/models/"

data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
models_folder = "G:\Meu Drive\mestrado_final_files\models\\"

In [ ]:
print(f"Pandas version: {pd.__version__},\nNumpy Version: {np.__version__},\
      \nTorch Version: {torch.__version__},\nMatplotlib Version: {plt.matplotlib.__version__}")

In [ ]:
pd.set_option("display.expand_frame_repr", False)

In [ ]:
with open(data_folder + 'dict_set_covariables.pkl', 'rb') as f:
    dict_set_covariables = pickle.load(f)

In [ ]:

df_games_nba = pd.read_csv(data_folder + "games_nba.csv", sep = ';')\
[['GAME_ID', 'HOME_TEAM_ID', 'AWAY_TEAM_ID', 'HOME_TEAM', 'AWAY_TEAM', 'GAME_DATE', 'HOME_PTS', 'AWAY_PTS']]

columns_boxscore = ["home_FGM", "away_FGM", "home_FGA", "away_FGA", "home_FGP", "away_FGP", 
                    "home_3FGM", "away_3FGM", "home_3FGA", "away_3FGA", "home_3FGP", "away_3FGP", 
                    "home_FTM", "away_FTM", "home_FTA", "away_FTA", "home_FTP", "away_FTP", 
                    "home_RebOff", "away_RebOff", "home_RebDef", "away_RebDef", "home_RebTot", 
                    "away_RebTot", "home_AST", "away_AST", "home_STL", "away_STL", "home_BLK", 
                    "away_BLK", "home_TO", "away_TO", "home_FP", "away_FP", "home_points", 
                    "away_points", "home_PMP", "away_PMP", "home_outcome", 'away_outcome']

df_boxscore = pd.read_csv(data_folder + 'boxscore_moving_average20.txt', sep = ';')\
.set_index("game_id")\
[columns_boxscore]

In [ ]:
tokens_info = pd.read_csv(data_folder + 'tokens_info.txt', sep = ';')\
.sort_values(['cod', 'token'])\
.set_index('token')

# Convert the index of tokens_info to a list of tokens
list_tokens = tokens_info.index.to_list()

# Create a dictionary mapping each token to a unique integer, starting from 1
token_to_integer = {tok: idx + 1 for idx, tok in enumerate(list_tokens)}

# Assign a special integer (0) for the period (.)
token_to_integer['.'] = 0

# Create an inverse mapping from integers back to tokens
integer_to_token = {idx: tok for tok, idx in token_to_integer.items()}

# tokens_info = tokens_info\
# .query("hpts != 0 or vpts != 0 or limhteam != 0 or limvteam != 0")\
# .to_dict(orient = 'index')


In [ ]:
%run 02_functions_simulations.ipynb

In [ ]:
VOCAB_SIZE = 601
TIME_SIZE = 301
CONTEXT_SIZE = 5
EMBEDDING_LAYER_SIZE = 64
NUM_HIDDEN_NEURONS = 
NUM_LAYERS = 
dropout_rate = 
layer_norm = 

In [ ]:
index_covariables = dict_set_covariables["index"][1]

In [ ]:
model_file = models_folder + 'model_ngram_tokens_new_variables_parameters.pth'

token_model = NGramModel_ExtraInfo(
    context_size = CONTEXT_SIZE, 
    vocab_size = VOCAB_SIZE, 
    embedding_dim = EMBEDDING_LAYER_SIZE,
    numerical_input_dim = len(index_covariables), 
    num_hidden_neurons = NUM_HIDDEN_NEURONS,
    num_layers = NUM_LAYERS, 
    activation = nn.GELU, 
    dropout_rate = dropout_rate, 
    use_layer_norm = layer_norm)

token_model.load_state_dict(torch.load(model_file, weights_only = True, map_location = torch.device('cpu')))
token_model.eval()

In [ ]:
model_file = models_folder + 'model_ngram_time_new_variables_parameters.pth'

time_model = SpentTimeModel(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, 
                            time_size = TIME_SIZE,
                            embedding_dim = EMBEDDING_LAYER_SIZE, 
                            numerical_input_dim = len(index_covariables),
                            num_hidden_neurons = NUM_HIDDEN_NEURONS, 
                            num_layers = NUM_LAYERS,
                            activation_function = nn.GELU, dropout_rate = dropout_rate, 
                            use_layer_norm = layer_norm)

time_model.load_state_dict(torch.load(model_file, weights_only = True, map_location = torch.device('cpu')))
time_model.eval()

In [ ]:
xdf0 = generate_game_plays(token_model = token_model, 
                           time_model = time_model, 
                           num_simulations = 10000, 
                           token_context = 5, 
                           num_numerical_variables = 14, 
                           config = 1)